# 02 Human in the Loop
In this notebook we'll learn how to add supervision to our Agent with a "Human in the Loop". Where should we put the Human? After all the goal of the agent is to automate. So where to put the human is just as important a decision as what tools to give our agent, or which model should we choose.

We can boil down the many use-cases for "Human in the Loop" to 3 main categories:
1. Sensitive Actions: approving sensitive actions, such as making claims disbursements for a Claims Agent.
2. Adding missing content - 
3. Debugging our Agent - 

In this notebook we'll cover the 1st category "Approving Sensitive Actions". The techniques we discuss here are the exact same techniques you'd use for the two other types of use-cases.

Let's say I have an Email inbox assistant, which can read email in my in-box and can summarize emails for me - nothing too risky about this so far. But let's say it can also draft and send emails on my behalf - this is maybe an action whare you'd like to have the final say (before the email is actually sent). The example below does just that.

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

Below is a _dummy_ email reader

In [2]:
from langchain.tools import tool, ToolRuntime


@tool
def read_email(runtime: ToolRuntime) -> str:
    """read an email from given email address"""
    # for thie example we'll take email from state
    return runtime.state["email"]


@tool
def send_email(body: str) -> str:
    """sends the email to a given address with a given subject & email body"""
    # code for sending email here...
    return "Email Sent!"

In [7]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware


class EmailState(AgentState):
    email: str


agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                # send_email tool required Human Approval
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        )
    ],
)

In [11]:
from langchain.messages import HumanMessage
from pprint import pprint

config = {"configurable": {"thread_id": "htil_id1"}}

response = agent.invoke(
    {
        "messages": [
            HumanMessage("Please read the email & send an appropriate response")
        ],
        "email": "Hi Bilbo, I'm gonna be late for our meeting tomorrow. Can we re-schedule? Best, Frodo",
    },
    config=config,
)

In [14]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Subject: '
                                                                          'Re: '
                                                                          'Meeting '
                                                                          'tomorrow\n'
                                                                          '\n'
                                                                          'Hi '
                                                                          'Frodo,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
    

In [18]:
response["messages"][-1]

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1054, 'prompt_tokens': 205, 'total_tokens': 1259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DXoZddr3Ts6tiLoKzmLV7iTfsvwPl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dba91-a5a2-7de2-8dc6-89c8200cf592-0', tool_calls=[{'name': 'send_email', 'args': {'body': 'Subject: Re: Meeting tomorrow\n\nHi Frodo,\n\nNo problem—thanks for the heads-up. We can reschedule. What time tomorrow works for you? I’m available at 10:00 AM or 2:00 PM, or let me know a time that’s convenient and I’ll adapt.\n\nBest,\nBilbo'}, 'id': 'call_C9yHPic4j1XXupQuMPHO8DyS', 'type': 'tool_